In [0]:
# ============================================================
# SHOPSTREAM LAKEHOUSE — BRONZE LAYER
# Notebook : 01_bronze_ingestion
# Source   : Olist Brazilian E-Commerce Dataset
# Purpose  : Ingest raw CSVs → Delta Lake Bronze tables
# Author   : El Mahdi Jamrani
# ============================================================
from datetime import timezone
from pyspark.sql import functions as F
from datetime import datetime

# All 9 source files live here
VOLUME_PATH = "/Volumes/workspace/default/raw_data"
BRONZE_PATH = "/Volumes/workspace/default/bronze"

INGESTION_TIMESTAMP = datetime.now(timezone.utc).isoformat()
SOURCE_SYSTEM       = "olist_ecommerce_kaggle"

print(f"Source : {VOLUME_PATH}")
print(f"Bronze : {BRONZE_PATH}")
print(f"Run at : {INGESTION_TIMESTAMP}")

Source : /Volumes/workspace/default/raw_data
Bronze : /Volumes/workspace/default/bronze
Run at : 2026-06-28T11:57:39.913755+00:00


In [0]:
def ingest_to_bronze(filename: str, table_name: str):
    """
    Reads one raw CSV from the Volume,
    adds audit columns, writes as Delta table.
    Rule: NO transformations here. Bronze = raw + metadata only.
    """
    source_path = f"{VOLUME_PATH}/{filename}"
    output_path = f"{BRONZE_PATH}/{table_name}"

    print(f"\n{'='*55}")
    print(f"  Ingesting : {filename}")

    df = (spark.read
               .option("header", "true")
               .option("inferSchema", "true")
               .option("multiLine", "true")
               .option("escape", '"')
               .csv(source_path))

    raw_count = df.count()
    print(f"  Raw rows  : {raw_count:,}")

    # Audit columns — mandatory on every Bronze table
    df = (df
          .withColumn("_source_file",         F.lit(source_path))
          .withColumn("_source_system",       F.lit(SOURCE_SYSTEM))
          .withColumn("_ingestion_timestamp", F.lit(INGESTION_TIMESTAMP))
          .withColumn("_ingestion_date",      F.current_date()))

    # Write as Delta — overwrite makes this idempotent
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .save(output_path))

    print(f"  Written to: {output_path}")
    return raw_count

In [0]:
# Create the bronze volume if it doesn't exist
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.bronze")
print("✅ Bronze volume ready.")

✅ Bronze volume ready.


In [0]:
tables = {
    "olist_customers_dataset.csv"            : "customers",
    "olist_orders_dataset.csv"               : "orders",
    "olist_order_items_dataset.csv"          : "order_items",
    "olist_order_payments_dataset.csv"       : "order_payments",
    "olist_order_reviews_dataset.csv"        : "order_reviews",
    "olist_products_dataset.csv"             : "products",
    "olist_sellers_dataset.csv"              : "sellers",
    "olist_geolocation_dataset.csv"          : "geolocation",
    "product_category_name_translation.csv"  : "product_category_translation",
}

results = {}
for filename, table_name in tables.items():
    results[table_name] = ingest_to_bronze(filename, table_name)

print("\n✅ All Bronze tables ingested.")


  Ingesting : olist_customers_dataset.csv
  Raw rows  : 99,441
  Written to: /Volumes/workspace/default/bronze/customers

  Ingesting : olist_orders_dataset.csv
  Raw rows  : 99,441
  Written to: /Volumes/workspace/default/bronze/orders

  Ingesting : olist_order_items_dataset.csv
  Raw rows  : 112,650
  Written to: /Volumes/workspace/default/bronze/order_items

  Ingesting : olist_order_payments_dataset.csv
  Raw rows  : 103,886
  Written to: /Volumes/workspace/default/bronze/order_payments

  Ingesting : olist_order_reviews_dataset.csv
  Raw rows  : 99,224
  Written to: /Volumes/workspace/default/bronze/order_reviews

  Ingesting : olist_products_dataset.csv
  Raw rows  : 32,951
  Written to: /Volumes/workspace/default/bronze/products

  Ingesting : olist_sellers_dataset.csv
  Raw rows  : 3,095
  Written to: /Volumes/workspace/default/bronze/sellers

  Ingesting : olist_geolocation_dataset.csv
  Raw rows  : 1,000,163
  Written to: /Volumes/workspace/default/bronze/geolocation

  Ing

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

for table_name in tables.values():
    output_path = f"{BRONZE_PATH}/{table_name}"
    spark.sql(f"""
        CREATE OR REPLACE VIEW workspace.bronze.{table_name}
        AS SELECT * FROM delta.`{output_path}`
    """)
    print(f"  Registered: workspace.bronze.{table_name}")

print("\n✅ All tables registered in workspace.bronze schema.")

  Registered: workspace.bronze.customers
  Registered: workspace.bronze.orders
  Registered: workspace.bronze.order_items
  Registered: workspace.bronze.order_payments
  Registered: workspace.bronze.order_reviews
  Registered: workspace.bronze.products
  Registered: workspace.bronze.sellers
  Registered: workspace.bronze.geolocation
  Registered: workspace.bronze.product_category_translation

✅ All tables registered in workspace.bronze schema.


In [0]:
%sql
SELECT 'customers'                    AS table_name, COUNT(*) AS rows FROM workspace.bronze.customers                   UNION ALL
SELECT 'orders'                       AS table_name, COUNT(*) AS rows FROM workspace.bronze.orders                      UNION ALL
SELECT 'order_items'                  AS table_name, COUNT(*) AS rows FROM workspace.bronze.order_items                 UNION ALL
SELECT 'order_payments'               AS table_name, COUNT(*) AS rows FROM workspace.bronze.order_payments              UNION ALL
SELECT 'order_reviews'                AS table_name, COUNT(*) AS rows FROM workspace.bronze.order_reviews               UNION ALL
SELECT 'products'                     AS table_name, COUNT(*) AS rows FROM workspace.bronze.products                    UNION ALL
SELECT 'sellers'                      AS table_name, COUNT(*) AS rows FROM workspace.bronze.sellers                     UNION ALL
SELECT 'geolocation'                  AS table_name, COUNT(*) AS rows FROM workspace.bronze.geolocation                 UNION ALL
SELECT 'product_category_translation' AS table_name, COUNT(*) AS rows FROM workspace.bronze.product_category_translation
ORDER BY rows DESC

table_name,rows
geolocation,1000163
order_items,112650
order_payments,103886
customers,99441
orders,99441
order_reviews,99224
products,32951
sellers,3095
product_category_translation,71
